# Suzhou Urban Expansion Simulation with UrbanM2M

This notebook reproduces a Suzhou urban expansion experiment with the OpenGMS UrbanM2M model. The workflow follows a normal modeling process: inspect the study data, prepare the UrbanM2M inputs, keep the model invocation parameters explicit, and analyze the simulated 2013 expansion result.

## 1. Background and Research Objectives

Urban expansion is a direct result of urbanization. Simulating future land-use expansion helps planners understand how a city may grow under observed historical patterns and spatial constraints.

This case uses the UrbanM2M model to simulate urban expansion in Suzhou. The experiment uses historical land-use rasters, spatial driving variables, and a cached OpenGMS model result for reproducible analysis in OpenGeoLab.

## 2. Data and Environment

The case data are stored as normal project files under `data/`. The main model input package is `data/Suzhou.zip`, which contains the research range, restriction layer, historical land-use rasters, and spatial variables used by UrbanM2M.

Key paths used in this notebook:

- `data/Suzhou-Masked/year/land_2010.tif`
- `data/Suzhou-Masked/year/land_2013.tif`
- `data/Result/sim_2013.tif`
- `data/Result/prob_2013.tif`
- `data/Suzhou.zip`


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap
from pygeomodel import GeoModeler

DATA_DIR = Path("data")
MODEL_ID = "d65631a5-9e1a-4450-9487-640c5a6494c2"
MODEL_NAME = "UrbanM2M城市扩张模拟模型"

suzhou_zip = DATA_DIR / "Suzhou.zip"
range_path = DATA_DIR / "Suzhou-Masked" / "range.tif"
restriction_path = DATA_DIR / "Suzhou-Masked" / "restriction.tif"
slope_path = DATA_DIR / "Suzhou-Masked" / "vars0" / "slope.tif"
town_path = DATA_DIR / "Suzhou-Masked" / "vars0" / "town.tif"
county_path = DATA_DIR / "Suzhou-Masked" / "vars0" / "county.tif"
land_2010_path = DATA_DIR / "Suzhou-Masked" / "year" / "land_2010.tif"
land_2013_path = DATA_DIR / "Suzhou-Masked" / "year" / "land_2013.tif"
sim_2013_path = DATA_DIR / "Result" / "sim_2013.tif"
prob_2013_path = DATA_DIR / "Result" / "prob_2013.tif"

boundary_path = DATA_DIR / "Suzhou" / "sz.shp"
region_path = DATA_DIR / "Around-Taihu-Lake" / "yrdbuffer.shp"

## 3. Load the Study Area

The Suzhou boundary is used to understand the study region. The model rasters are already clipped to the research area in `Suzhou-Masked`.

In [ ]:
suzhou = gpd.read_file(boundary_path)
region = gpd.read_file(region_path)

fig, ax = plt.subplots(figsize=(6, 6))
region.to_crs(suzhou.crs).boundary.plot(ax=ax, color="#9aa4b2", linewidth=1.2, label="Taihu region buffer")
suzhou.boundary.plot(ax=ax, color="#0f172a", linewidth=1.5, label="Suzhou")
ax.set_title("Study Area: Suzhou and Around-Taihu-Lake Context")
ax.set_axis_off()
ax.legend(loc="lower right")

## 4. Inspect Historical Urban Land and Spatial Variables

UrbanM2M uses historical land-use rasters and spatial variables to learn urban expansion patterns. Here we compare the 2010 and 2013 urban maps and inspect the slope driver.

In [ ]:
with rasterio.open(land_2010_path) as src:
    land_2010 = src.read(1, masked=True)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

with rasterio.open(land_2013_path) as src:
    land_2013 = src.read(1, masked=True)

with rasterio.open(slope_path) as src:
    slope = src.read(1, masked=True)

land_cmap = ListedColormap(["#f8fafc", "#26384d"])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(land_2010, cmap=land_cmap, interpolation="nearest")
axes[0].set_title("Urban Land 2010")
axes[1].imshow(land_2013, cmap=land_cmap, interpolation="nearest")
axes[1].set_title("Urban Land 2013")
im = axes[2].imshow(slope, cmap="terrain", interpolation="nearest")
axes[2].set_title("Slope Variable")
for ax in axes:
    ax.set_axis_off()
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()

urban_2010 = int((land_2010.compressed() == 1).sum())
urban_2013 = int((land_2013.compressed() == 1).sum())
print(f"Urban pixels in 2010: {urban_2010:,}")
print(f"Urban pixels in 2013: {urban_2013:,}")
print(f"Observed growth from 2010 to 2013: {urban_2013 - urban_2010:,} pixels")

## 5. Configure the UrbanM2M Model

The OpenGMS model item is `d65631a5-9e1a-4450-9487-640c5a6494c2`. The parameters below mirror the original experiment: 2010 is the start year, 2013 is the first simulated year, and the simulation length is one year.

In [ ]:
modeler = GeoModeler()

params = {
    "Get_research_range": str(range_path),
    "Get_research_restriction": str(restriction_path),
    "Get_history_land_data": str(suzhou_zip),
    "Get_slope_data": str(slope_path),
    "Get_dist_town_data": str(town_path),
    "Get_dist_county_data": str(county_path),
    "st_year": 2010,
    "first_sim_year": 2013,
    "out_len": 1,
    "land_demands": "1000",
}

params

## 6. Invoke the OpenGMS Model

The result files included with this case were produced by the UrbanM2M service. When the OpenGeoLab runtime is configured for live OpenGMS execution, run the cell below to submit a new task. The rest of the notebook reads the included result so the analysis remains reproducible.

In [ ]:
# Run this cell when the runtime is configured for OpenGMS model execution.
# live_result = modeler.invoke(
#     MODEL_NAME,
#     params=params,
#     output_dir=Path("data/Result/live_run")
# )
# live_result.save(output_dir="data/Result/live_run")
# live_result

## 7. Read the Simulated 2013 Result

UrbanM2M produces a simulated urban land raster and a transition probability raster. This case keeps the cached 2013 result in `data/Result`.

In [ ]:
with rasterio.open(sim_2013_path) as src:
    sim_2013 = src.read(1, masked=True)

with rasterio.open(prob_2013_path) as src:
    prob_2013 = src.read(1, masked=True)

sim_urban = int((sim_2013.compressed() == 1).sum())
print(f"Simulated urban pixels in 2013: {sim_urban:,}")
print(f"Probability range: {float(prob_2013.min()):.4f} - {float(prob_2013.max()):.4f}")

## 8. Visualize Simulated Expansion and Probability

The simulated expansion map compares the 2010 base state with the simulated 2013 urban map. Newly simulated urban pixels are highlighted in red.

In [ ]:
base = np.ma.filled(land_2010, 0)
simulated = np.ma.filled(sim_2013, 0)

change_map = np.zeros_like(base)
change_map[base == 1] = 1
change_map[(base == 0) & (simulated == 1)] = 2

change_cmap = ListedColormap(["#f8fafc", "#26384d", "#ef4444"])
legend_items = [
    mpatches.Patch(color="#26384d", label="Urban in 2010"),
    mpatches.Patch(color="#ef4444", label="New simulated expansion"),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(change_map, cmap=change_cmap, interpolation="nearest")
axes[0].set_title("Simulated Expansion: 2010 to 2013")
axes[0].legend(handles=legend_items, loc="lower right")
prob_plot = axes[1].imshow(prob_2013, cmap="magma", interpolation="nearest")
axes[1].set_title("Urban Transition Probability")
for ax in axes:
    ax.set_axis_off()
fig.colorbar(prob_plot, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()

## 9. Validate Against Observed 2013 Urban Land

The validation uses the overlapping valid area of the observed 2013 raster and the simulated 2013 raster. This keeps the statistics consistent with the model output mask.

In [ ]:
valid_mask = (~land_2013.mask) & (~sim_2013.mask)
truth = land_2013.data[valid_mask]
prediction = sim_2013.data[valid_mask]

validation_map = np.zeros_like(base)
validation_map[(np.ma.filled(land_2013, 0) == 1) & (simulated == 1)] = 1
validation_map[(np.ma.filled(land_2013, 0) == 1) & (simulated == 0)] = 2
validation_map[(np.ma.filled(land_2013, 0) == 0) & (simulated == 1)] = 3
validation_map[~valid_mask] = 0

correct = int(((truth == 1) & (prediction == 1)).sum())
missed = int(((truth == 1) & (prediction == 0)).sum())
false_alarm = int(((truth == 0) & (prediction == 1)).sum())

precision = correct / (correct + false_alarm)
recall = correct / (correct + missed)
f1 = 2 * precision * recall / (precision + recall)

validation_cmap = ListedColormap(["#ffffff", "#3498db", "#f1c40f", "#c0392b"])
validation_legend = [
    mpatches.Patch(color="#3498db", label="Hit"),
    mpatches.Patch(color="#f1c40f", label="Miss"),
    mpatches.Patch(color="#c0392b", label="False alarm"),
]

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(validation_map, cmap=validation_cmap, interpolation="nearest")
ax.set_title("UrbanM2M Validation: Observed vs. Simulated 2013")
ax.legend(handles=validation_legend, loc="lower right")
ax.set_axis_off()

print(f"Correct urban pixels: {correct:,}")
print(f"Missed urban pixels: {missed:,}")
print(f"False alarm pixels: {false_alarm:,}")
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1-score: {f1:.4f}")

## 10. Urban Expansion Statistics

The raster resolution is 30 m, so each pixel represents 0.0009 km². The table below summarizes the simulated expansion between 2010 and 2013.

In [ ]:
pixel_area_km2 = (30 * 30) / 1_000_000
area_2010 = urban_2010 * pixel_area_km2
area_sim_2013 = sim_urban * pixel_area_km2
new_expansion_pixels = int(((base == 0) & (simulated == 1) & valid_mask).sum())
new_expansion_area = new_expansion_pixels * pixel_area_km2
expansion_rate = new_expansion_area / area_2010

print("Urban Expansion Simulation Report")
print("---------------------------------")
print(f"Initial urban area in 2010: {area_2010:,.2f} km²")
print(f"Simulated urban area in 2013: {area_sim_2013:,.2f} km²")
print(f"New simulated expansion: {new_expansion_area:,.2f} km²")
print(f"Expansion rate: {expansion_rate:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1-score: {f1:.4f}")

## 11. Conclusion

UrbanM2M captures the major spatial pattern of Suzhou urban expansion between 2010 and 2013. The simulated urban map has high precision, meaning most predicted urban pixels are consistent with the observed 2013 urban land. The recall shows that some observed growth is still underestimated, which is useful for discussing model uncertainty and the influence of demand settings.